In [1]:
from model_wrapper import *
import cuml.accel
from math import floor
cuml.accel.install()

Current RAM usage: 298.77 MB


In [2]:
from sklearn.decomposition import IncrementalPCA

def data_pipeline(plane):
    _x_train, _y_train = get_data(train_full, "train_series", plane)
    _x_test, _y_test = get_data(test_full, "train_series", plane)

    _x_train = np.reshape(_x_train, shape=(_x_train.shape[0], _x_train.shape[1] * _x_train.shape[2] *  _x_train.shape[3]))
    _x_test = np.reshape(_x_test, shape=(_x_test.shape[0], _x_test.shape[1] * _x_test.shape[2] *  _x_test.shape[3]))

    n_components = 15
    n_batches = floor(_x_train.shape[0] / n_components)
    inc_pca = IncrementalPCA(n_components=n_components)

    for X_batch in np.array_split(_x_train, n_batches):
        inc_pca.partial_fit(X_batch)

    _x_train = inc_pca.transform(_x_train)
    _x_test = inc_pca.transform(_x_test)
    
    return _x_train, _y_train, _x_test, _y_test

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
planes = ["Sagittal", "Axial", "Coronal"]

for p in planes:
    x_train, y_train, x_test, y_test = data_pipeline(p)

    model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    model.fit(x_train, y_train)
    pred = model.predict(x_test)

    scores = roc_auc_score(y_test, pred, average=None)
    print(f"{p} Model Scores: {np.mean(scores)}")
    for i in range(len(target_columns)):
        print(f"\t{target_columns[i]}: {scores[i]}")
    print("--------")


Sagittal Model Scores: 0.5760435166685166
	ACL: 0.6142857142857143
	MCL: 0.5
	Medial Meniscus: 0.8333333333333333
	Lateral Meniscus: 0.6071428571428572
	Medial OA: 0.4326923076923077
	Lateral OA: 0.6166666666666667
	PF OA: 0.7211538461538461
	Effusion: 0.4652777777777778
	Synovitis: 0.6136363636363635
	Baker's: 0.5499999999999999
	Contusion: 0.4583333333333333
	Fracture: 0.5
--------
Axial Model Scores: 0.5451388888888888
	ACL: 0.5
	MCL: 0.5
	Medial Meniscus: 0.6666666666666666
	Lateral Meniscus: 0.35
	Medial OA: 0.575
	Lateral OA: 0.4166666666666667
	PF OA: 0.875
	Effusion: 0.325
	Synovitis: 0.8333333333333333
	Baker's: 0.5
	Contusion: 0.5
	Fracture: 0.5
--------
Coronal Model Scores: 0.4916666666666667
	ACL: 0.3857142857142857
	MCL: 0.5
	Medial Meniscus: 0.6857142857142857
	Lateral Meniscus: 0.17142857142857146
	Medial OA: 0.5857142857142856
	Lateral OA: 0.3
	PF OA: 0.7285714285714286
	Effusion: 0.8
	Synovitis: 0.44285714285714284
	Baker's: 0.3
	Contusion: 0.5
	Fracture: 0.5
--------

In [ ]:
class LogRegModel(Model):
    pass